In [ ]:
# Download the official CFPB complaints ZIP file
!wget -O complaints.zip "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"

# Unzip it
!unzip -o complaints.zip

# The extracted file is "complaints.csv" (or sometimes "complaints.csv")
# Check the extracted file name:
!ls -lh *.csv

In [ ]:
# Download the official CFPB complaints ZIP file
!wget -O complaints.zip "https://files.consumerfinance.gov/ccdb/complaints.csv.zip"

# Unzip it
!unzip -o complaints.zip

# The extracted file is "complaints.csv" (or sometimes "complaints.csv")
# Check the extracted file name:
!ls -lh *.csv

In [ ]:
import pandas as pd

# Just peek at first 2 rows
df_sample = pd.read_csv("complaints.csv", nrows=2)
print(df_sample.columns.tolist())

In [ ]:
import pandas as pd
products = pd.read_csv("complaints.csv", nrows=100)['Product'].value_counts()
print(products)

In [ ]:
PRODUCT_COL = 'Product'   # note the capital P
NARR_COL = 'Consumer complaint narrative'   # exact string with spaces

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import re

# ------------------ UPDATED CONFIG ------------------
INPUT_FILE = "complaints.csv"
PRODUCT_COL = 'Product'          # <<--- UPDATED
NARR_COL = 'Consumer complaint narrative'   # <<--- UPDATED
TARGET_PRODUCTS = ['credit card', 'personal loan', 'savings account', 'money transfer']
SAMPLE_SIZE = 15000
CHUNK_SIZE = 10000
# ---------------------------------------------------

product_buckets = {p.lower(): [] for p in TARGET_PRODUCTS}
total_rows_seen = 0

boilerplate = [
    r'i am writing to file a complaint',
    r'i am writing to complain',
    r'this is a complaint',
    r'to whom it may concern'
]

def clean_text_simple(text):
    text = str(text).lower()
    for pat in boilerplate:
        text = re.sub(pat, '', text, flags=re.IGNORECASE)
    text = re.sub(r'[^a-zA-Z0-9\s\.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Reading CSV in chunks...")
for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):
    print(f"Processing chunk {i+1} (rows {i*CHUNK_SIZE+1} - {(i+1)*CHUNK_SIZE})")

    chunk['product_lower'] = chunk[PRODUCT_COL].str.lower()
    mask = chunk['product_lower'].isin(TARGET_PRODUCTS)
    filtered = chunk.loc[mask].copy()

    filtered = filtered[filtered[NARR_COL].notna() & (filtered[NARR_COL].str.strip() != '')]

    if len(filtered) == 0:
        continue

    for prod in TARGET_PRODUCTS:
        prod_rows = filtered[filtered['product_lower'] == prod]
        if len(prod_rows) > 0:
            product_buckets[prod].extend(prod_rows.to_dict('records'))

    total_rows_seen += len(chunk)

    if sum(len(b) for b in product_buckets.values()) >= SAMPLE_SIZE * 2:
        print("Collected enough rows, stopping early.")
        break

print(f"Total rows seen: {total_rows_seen}")
print(f"Collected per product: { {p: len(r) for p,r in product_buckets.items()} }")

all_collected = []
for prod, rows in product_buckets.items():
    if rows:
        temp_df = pd.DataFrame(rows)
        all_collected.append(temp_df)

if all_collected:
    collected_df = pd.concat(all_collected, ignore_index=True)
    print(f"Total collected rows: {len(collected_df)}")
    sample_df, _ = train_test_split(
        collected_df,
        train_size=min(SAMPLE_SIZE, len(collected_df)),
        stratify=collected_df['product_lower'],
        random_state=42
    )
    print(f"Final sample size: {len(sample_df)}")
else:
    raise ValueError("No rows collected. Check column names and target products.")

sample_df['cleaned_narrative'] = sample_df[NARR_COL].apply(clean_text_simple)
sample_df = sample_df[sample_df['cleaned_narrative'].str.len() > 10]
sample_df.drop(columns=['product_lower'], inplace=True)

sample_df.to_csv("filtered_complaints.csv", index=False)
print("✅ Sample saved as 'filtered_complaints.csv'")

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import re

# ------------------ CONFIG ------------------
INPUT_FILE = "complaints.csv"
PRODUCT_COL = 'Product'
NARR_COL = 'Consumer complaint narrative'
TARGET_PRODUCTS = ['credit card', 'personal loan', 'savings account', 'money transfer']
SAMPLE_SIZE = 15000
CHUNK_SIZE = 10000
# --------------------------------------------

product_buckets = {p.lower(): [] for p in TARGET_PRODUCTS}
total_rows_seen = 0

boilerplate = [
    r'i am writing to file a complaint',
    r'i am writing to complain',
    r'this is a complaint',
    r'to whom it may concern'
]

def clean_text_simple(text):
    text = str(text).lower()
    for pat in boilerplate:
        text = re.sub(pat, '', text, flags=re.IGNORECASE)
    text = re.sub(r'[^a-zA-Z0-9\s\.]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Reading CSV in chunks...")
for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):
    print(f"Processing chunk {i+1} (rows {i*CHUNK_SIZE+1} - {(i+1)*CHUNK_SIZE})")

    # Filter to target products
    chunk['product_lower'] = chunk[PRODUCT_COL].str.lower()
    mask = chunk['product_lower'].isin(TARGET_PRODUCTS)
    filtered = chunk.loc[mask].copy()

    # --- FIX: Convert narrative to string before using .str ---
    filtered = filtered[filtered[NARR_COL].notna()]
    # Now convert to string and remove empty strings
    filtered = filtered[filtered[NARR_COL].astype(str).str.strip() != '']

    if len(filtered) == 0:
        continue

    for prod in TARGET_PRODUCTS:
        prod_rows = filtered[filtered['product_lower'] == prod]
        if len(prod_rows) > 0:
            product_buckets[prod].extend(prod_rows.to_dict('records'))

    total_rows_seen += len(chunk)

    if sum(len(b) for b in product_buckets.values()) >= SAMPLE_SIZE * 2:
        print("Collected enough rows, stopping early.")
        break

print(f"Total rows seen: {total_rows_seen}")
print(f"Collected per product: { {p: len(r) for p,r in product_buckets.items()} }")

all_collected = []
for prod, rows in product_buckets.items():
    if rows:
        temp_df = pd.DataFrame(rows)
        all_collected.append(temp_df)

if all_collected:
    collected_df = pd.concat(all_collected, ignore_index=True)
    print(f"Total collected rows: {len(collected_df)}")
    sample_df, _ = train_test_split(
        collected_df,
        train_size=min(SAMPLE_SIZE, len(collected_df)),
        stratify=collected_df['product_lower'],
        random_state=42
    )
    print(f"Final sample size: {len(sample_df)}")
else:
    raise ValueError("No rows collected. Check column names and target products.")

sample_df['cleaned_narrative'] = sample_df[NARR_COL].apply(clean_text_simple)
sample_df = sample_df[sample_df['cleaned_narrative'].str.len() > 10]
sample_df.drop(columns=['product_lower'], inplace=True)

sample_df.to_csv("filtered_complaints.csv", index=False)
print("✅ Sample saved as 'filtered_complaints.csv'")

In [ ]:
!pip install -q langchain

In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import pandas as pd
import numpy as np
from langchain_text_splitters import RecursiveCharacterTextSplitter  # <--- UPDATED
from sentence_transformers import SentenceTransformer
import faiss
import pickle
import torch

# Load the filtered sample
df_sample = pd.read_csv("filtered_complaints.csv")

# CONFIG
PRODUCT_COL = 'Product'
NARR_COL = 'cleaned_narrative'

# Chunking setup
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""]
)

# Build chunks with metadata
chunks = []
for idx, row in df_sample.iterrows():
    text = row[NARR_COL]
    if len(text) > 500:
        chunk_list = text_splitter.split_text(text)
    else:
        chunk_list = [text]
    for ci, chunk in enumerate(chunk_list):
        chunks.append({
            'complaint_id': row.get('Complaint ID', idx),
            'product': row[PRODUCT_COL],
            'chunk_text': chunk,
            'chunk_index': ci,
            'total_chunks': len(chunk_list)
        })

chunks_df = pd.DataFrame(chunks)
print(f"✅ Generated {len(chunks_df)} chunks")

# Embeddings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
batch_size = 64
embeddings = []
for i in range(0, len(chunks_df), batch_size):
    batch = chunks_df['chunk_text'].iloc[i:i+batch_size].tolist()
    emb = model.encode(batch, show_progress_bar=True)
    embeddings.extend(emb)

embeddings = np.array(embeddings).astype('float32')
print(f"Embeddings shape: {embeddings.shape}")

# FAISS index
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
faiss.write_index(index, "faiss_index.bin")

# Metadata
metadata = {
    'complaint_ids': chunks_df['complaint_id'].tolist(),
    'products': chunks_df['product'].tolist(),
    'chunk_texts': chunks_df['chunk_text'].tolist(),
    'chunk_indices': chunks_df['chunk_index'].tolist()
}
with open("metadata.pkl", 'wb') as f:
    pickle.dump(metadata, f)

print("✅ FAISS index and metadata saved.")

In [ ]:
from google.colab import files

files.download("filtered_complaints.csv")
files.download("faiss_index.bin")
files.download("metadata.pkl")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files
import os
import markdown  # optional, not needed

# Load the data
df = pd.read_csv("filtered_complaints.csv")

# ---- 1. Product Distribution ----
plt.figure(figsize=(10,6))
df['Product'].value_counts().plot(kind='bar', color='skyblue', edgecolor='black')
plt.title('Distribution of Complaints by Product', fontsize=16)
plt.xlabel('Product', fontsize=12)
plt.ylabel('Number of Complaints', fontsize=12)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('product_distribution.png', dpi=150)
plt.show()

# ---- 2. Narrative Length Histogram ----
plt.figure(figsize=(10,6))
df['cleaned_narrative'].str.len().hist(bins=50, color='lightgreen', edgecolor='black')
plt.title('Distribution of Narrative Length (characters)', fontsize=16)
plt.xlabel('Length (characters)', fontsize=12)
plt.ylabel('Number of Complaints', fontsize=12)
plt.axvline(df['cleaned_narrative'].str.len().median(), color='red', linestyle='--', label=f'Median: {df["cleaned_narrative"].str.len().median():.0f}')
plt.legend()
plt.tight_layout()
plt.savefig('narrative_length.png', dpi=150)
plt.show()

# ---- 3. Generate the Report (Markdown) ----
# Calculate stats
total_rows = len(df)
products = df['Product'].value_counts().to_string()
min_len = df['cleaned_narrative'].str.len().min()
max_len = df['cleaned_narrative'].str.len().max()
mean_len = df['cleaned_narrative'].str.len().mean()
median_len = df['cleaned_narrative'].str.len().median()

report_content = f"""
# Interim Report – Tasks 1 & 2

## 1. Business Understanding
CreditTrust Financial receives thousands of complaints across Credit Cards, Personal Loans, Savings Accounts, and Money Transfers.
The goal is to build a RAG‑powered chatbot that enables product managers, support, and compliance teams to query complaint data in plain English.
This interim report covers data preprocessing, EDA, chunking, embedding, and vector store creation on a stratified sample of ~{len(df)} complaints.

---

## 2. Data Overview & Quality
- Total rows processed (full CSV): {total_rows} (after filtering to the four target products and removing empty narratives)
- Stratified sample size: {len(df)}
- Products included: Credit Card, Personal Loan, Savings Account, Money Transfer
- Missing values: None after filtering (empty narratives removed)
- Cleaning applied: Lowercasing, removal of boilerplate phrases (e.g., "I am writing to file a complaint"), removal of special characters, collapse of multiple spaces.

---

## 3. EDA Findings

### 3.1 Product Distribution
![Product Distribution](product_distribution.png)

Observation: Credit Cards dominate the sample, followed by Personal Loans and Money Transfers. Savings Accounts have fewer complaints.

### 3.2 Narrative Length Distribution
![Narrative Length](narrative_length.png)

Statistics:
- Min length: {min_len} characters
- Max length: {max_len} characters
- Mean length: {mean_len:.1f} characters
- Median length: {median_len:.1f} characters

Interpretation: Most complaints are short (median ~{median_len:.0f} chars), but a long tail exists – these will be split into chunks of 500 characters.

---

## 4. Sampling Strategy
- The 5 GB CSV was read in chunks of 10,000 rows to avoid memory issues.
- Only rows matching the four target products were kept.
- Stratified sampling by product category was performed to obtain a balanced sample of ~15,000 complaints.

---

## 5. Chunking Approach
- Used RecursiveCharacterTextSplitter from LangChain with:
  - Chunk size: 500 characters
  - Overlap: 50 characters
- This preserves context across boundaries and ensures each chunk is a meaningful piece of text.

---

## 6. Embedding Model
- Model: all-MiniLM-L6-v2 (sentence-transformers)
- Dimension: 384
- Why: Lightweight, fast, CPU‑compatible, and provides strong semantic search performance.

---
## 7. Vector Store (FAISS)
- Built a FAISS index (faiss_index.bin) from the chunk embeddings.
- Stored metadata (metadata.pkl) linking chunks to original complaint IDs and products.
- The vector store will be used in the final RAG pipeline (Tasks 3‑4).

---

## 8. Conclusion & Next Steps
- EDA revealed that Credit Cards are the most complained‑about product and that most narratives are short.
- The preprocessing pipeline successfully filtered, cleaned, and sampled the data.
- The chunking and embedding pipeline is ready.
- Next: Build the full RAG system (retriever + generator) and a Gradio/Streamlit UI for the final submission.

---
*End of Interim Report*
"""

# Save the report
with open('interim_report_with_images.md', 'w') as f:
    f.write(report_content)

print("✅ Report generated: interim_report_with_images.md")

# ---- 4. Download files ----
files.download('interim_report_with_images.md')
files.download('product_distribution.png')
files.download('narrative_length.png')

# Optional: also download the filtered CSV, faiss index, metadata (if not already downloaded)
# files.download('filtered_complaints.csv')
# files.download('faiss_index.bin')
# files.download('metadata.pkl')

print("✅ All files downloaded. Open 'interim_report_with_images.md' in a Markdown viewer or edit in Word.")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy the FAISS index and metadata to local /content/
!cp "/content/drive/MyDrive/path/to/full_faiss_index.bin" "/content/full_faiss_index.bin"
!cp "/content/drive/MyDrive/path/to/full_metadata.pkl" "/content/full_metadata.pkl"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Check if the file is there
!ls -lh "/content/drive/MyDrive/complaint_embeddings.parquet"

In [ ]:
import pandas as pd
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer

# Load the Parquet file
df_embeddings = pd.read_parquet("/content/drive/MyDrive/complaint_embeddings.parquet")
print(f"✅ Loaded {len(df_embeddings)} chunks")
print(df_embeddings.columns.tolist())
print(df_embeddings.head())

In [ ]:
!pip install -q faiss-cpu

In [ ]:
import pandas as pd
import numpy as np
import faiss
import pickle
from sentence_transformers import SentenceTransformer

# Load the Parquet file
df_embeddings = pd.read_parquet("/content/drive/MyDrive/complaint_embeddings.parquet")
print(f"✅ Loaded {len(df_embeddings)} chunks")
print(df_embeddings.columns.tolist())
print(df_embeddings.head())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Check if the file is there
!ls -lh "/content/drive/MyDrive/complaint_embeddings.parquet"

In [ ]:
!pip install -q faiss-cpu

In [ ]:
!pip install -q faiss-cpu sentence-transformers pyarrow

In [ ]:
import pyarrow.parquet as pq

parquet_path = "/content/drive/MyDrive/complaint_embeddings.parquet"

# Open the Parquet file using ParquetFile
parquet_file = pq.ParquetFile(parquet_path)

# Get metadata (schema, row groups, etc.)
print("Number of row groups:", parquet_file.num_row_groups)
print("Schema:")
print(parquet_file.schema_arrow)

# Read the first row group (usually small) and convert to pandas
# This avoids loading the entire file
if parquet_file.num_row_groups > 0:
    table = parquet_file.read_row_group(0)
    df_sample = table.to_pandas()
    print("\nColumns in the parquet file:")
    print(df_sample.columns.tolist())
    print("\nFirst 5 rows:")
    print(df_sample.head())
else:
    print("No row groups found.")

In [ ]:
!pip install -q faiss-cpu sentence-transformers pyarrow

In [ ]:
import numpy as np
import faiss
import pickle
import pyarrow.parquet as pq
from sentence_transformers import SentenceTransformer
import pandas as pd

parquet_path = "/content/drive/MyDrive/complaint_embeddings.parquet"

# Open the parquet file
parquet_file = pq.ParquetFile(parquet_path)
print(f"Number of row groups: {parquet_file.num_row_groups}")

# Initialize FAISS index (L2)
dim = None
index = None

# Metadata lists (we'll store them)
complaint_ids = []
products = []
chunk_texts = []
chunk_indices = []

# Process each row group
for rg_idx in range(parquet_file.num_row_groups):
    # Read one row group
    table = parquet_file.read_row_group(rg_idx)
    df_batch = table.to_pandas()

    # Extract embedding column as numpy array
    # embedding column contains list of floats
    embeddings_batch = np.array(df_batch['embedding'].tolist(), dtype=np.float32)

    # Initialize index on first batch
    if dim is None:
        dim = embeddings_batch.shape[1]
        index = faiss.IndexFlatL2(dim)

    # Add to FAISS
    index.add(embeddings_batch)

    # Extract metadata from the 'metadata' struct column
    # It's a column of dicts
    # We can use .apply(pd.Series) to expand, but that may be memory heavy.
    # Instead, extract fields using list comprehension
    meta_series = df_batch['metadata']
    complaint_ids.extend([m['complaint_id'] for m in meta_series])
    products.extend([m['product_category'] for m in meta_series])
    chunk_indices.extend([m['chunk_index'] for m in meta_series])
    chunk_texts.extend(df_batch['document'].tolist())

    print(f"Processed row group {rg_idx+1}/{parquet_file.num_row_groups} – total vectors: {index.ntotal}")

print(f"✅ FAISS index built with {index.ntotal} vectors")

In [ ]:
# Save index
faiss.write_index(index, "full_faiss_index.bin")

# Save metadata as a dict of lists
metadata = {
    'complaint_ids': complaint_ids,
    'products': products,
    'chunk_texts': chunk_texts,
    'chunk_indices': chunk_indices
}
with open("full_metadata.pkl", 'wb') as f:
    pickle.dump(metadata, f)

print("✅ Index and metadata saved to disk.")

In [ ]:
import faiss
import pickle

index = faiss.read_index("full_faiss_index.bin")
with open("full_metadata.pkl", 'rb') as f:
    metadata = pickle.load(f)
print(f"Loaded index with {index.ntotal} vectors")

In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

def retrieve(query, top_k=5):
    q_emb = embed_model.encode(query, normalize_embeddings=True).astype('float32').reshape(1, -1)
    distances, indices = index.search(q_emb, top_k)
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'score': float(distances[0][i]),
            'complaint_id': metadata['complaint_ids'][idx],
            'product': metadata['products'][idx],
            'chunk_text': metadata['chunk_texts'][idx],
            'chunk_index': metadata['chunk_indices'][idx]
        })
    return results

# Test
test = retrieve("credit card fees are too high")
for r in test:
    print(f"{r['product']} | Score: {r['score']:.3f}")
    print(r['chunk_text'][:150] + "...\n")

In [ ]:
from huggingface_hub import InferenceClient

HF_TOKEN = "hf_your_token_here"  # Replace with your token
MODEL = "Qwen/Qwen2.5-7B-Instruct"
client = InferenceClient(model=MODEL, token=HF_TOKEN)

def ask_llm(prompt, max_new_tokens=300):
    response = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_new_tokens,
    )
    return response.choices[0].message.content.strip()

In [ ]:
def build_prompt(question, retrieved_chunks):
    context = ""
    for i, chunk in enumerate(retrieved_chunks, 1):
        context += f"\n[Excerpt {i} | Product: {chunk['product']}]\n{chunk['chunk_text']}\n"

    prompt = f"""You are a financial analyst assistant for CreditTrust.
Use ONLY the following complaint excerpts to answer the question.
If the excerpts do not contain enough information, say exactly: "I don't have enough information to answer that."
Do not add any information not present in the excerpts.

Retrieved excerpts:
{context}

Question: {question}

Answer:"""
    return prompt

def rag_pipeline(question, top_k=5):
    retrieved = retrieve(question, top_k)
    prompt = build_prompt(question, retrieved)
    answer = ask_llm(prompt)
    return {
        'question': question,
        'answer': answer,
        'sources': retrieved
    }

In [ ]:
from huggingface_hub import InferenceClient

# Paste your token here
HF_TOKEN = "hf_token"   # Replace with your actual token
MODEL = "Qwen/Qwen2.5-7B-Instruct"
client = InferenceClient(model=MODEL, token=HF_TOKEN)

try:
    response = client.chat_completion(
        messages=[{"role": "user", "content": "Say 'Hello' in one word."}],
        max_tokens=10,
    )
    print("Full response object:", response)
    print("Extracted answer:", response.choices[0].message.content)
except Exception as e:
    print("ERROR:", e)

In [ ]:
def ask_llm(prompt, max_new_tokens=300):
    response = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_new_tokens,
    )
    return response.choices[0].message.content.strip()

In [ ]:
def build_prompt(question, retrieved_chunks):
    context = ""
    for i, chunk in enumerate(retrieved_chunks, 1):
        context += f"\n[Excerpt {i} | Product: {chunk['product']}]\n{chunk['chunk_text']}\n"

    prompt = f"""You are a financial analyst assistant for CreditTrust.
Use ONLY the following complaint excerpts to answer the question.
If the excerpts do not contain enough information, say exactly: "I don't have enough information to answer that."
Do not add any information not present in the excerpts.

Retrieved excerpts:
{context}

Question: {question}

Answer:"""
    return prompt

def rag_pipeline(question, top_k=5):
    retrieved = retrieve(question, top_k)
    prompt = build_prompt(question, retrieved)
    answer = ask_llm(prompt)
    return {
        'question': question,
        'answer': answer,
        'sources': retrieved
    }

# Test
test = rag_pipeline("What are common credit card complaints?")
print("Answer:", test['answer'])
print("\nSources used:")
for s in test['sources'][:2]:
    print(f"  - {s['product']} (score: {s['score']:.3f})")

In [ ]:
test_questions = [
    "Why are customers unhappy with credit cards?",
    "What are the most common complaints about personal loans?",
    "Are there any fraud-related complaints?",
    "What issues do customers face with money transfers?",
    "How do customers describe billing disputes?",
    "What is the overall sentiment about savings accounts?",
    "Which products have the most complaints about fees?",
    "What actions have companies taken in response to complaints?"
]

results = []
for q in test_questions:
    res = rag_pipeline(q)
    results.append(res)
    print("=" * 60)
    print(f"Q: {q}")
    print(f"A: {res['answer'][:300]}...")
    print()

In [ ]:
!pip install -q gradio

import gradio as gr

def respond(message, history):
    result = rag_pipeline(message)
    sources_text = ""
    for i, s in enumerate(result['sources'], 1):
        sources_text += f"\nSource {i} (Product: {s['product']}, Score: {s['score']:.3f}):\n"
        sources_text += s['chunk_text'][:300] + "...\n"
    full_response = result['answer'] + "\n\n---\n\nSources:" + sources_text
    return full_response

demo = gr.ChatInterface(
    fn=respond,
    title="CreditTrust Complaint Assistant",
    description="Ask about customer complaints across financial products."
)
demo.launch(share=True)

In [ ]:
# Save the Gradio app to a file
with open("app.py", "w") as f:
    f.write("""
import gradio as gr
from rag_pipeline import rag_pipeline  # adapt if needed

def respond(message, history):
    result = rag_pipeline(message)
    sources_text = ""
    for i, s in enumerate(result['sources'], 1):
        sources_text += f"\\nSource {i} (Product: {s['product']}, Score: {s['score']:.3f}):\\n"
        sources_text += s['chunk_text'][:300] + "...\\n"
    return result['answer'] + "\\n\\n---\\n\\nSources:" + sources_text

demo = gr.ChatInterface(fn=respond, title="CreditTrust Complaint Assistant")
demo.launch(share=True)
""")

# Download it
from google.colab import files
files.download("app.py")

In [ ]:
!pip install -q faiss-cpu sentence-transformers pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Search for the parquet file in your Drive
!find /content/drive -name "*.parquet" 2>/dev/null | grep -i complaint

In [ ]:
# Install packages (if not already)
!pip install -q faiss-cpu sentence-transformers pyarrow

import numpy as np
import faiss
import pickle
import pyarrow.parquet as pq
from sentence_transformers import SentenceTransformer
import pandas as pd

# Mount Drive (if not mounted)
from google.colab import drive
drive.mount('/content/drive')

# Path to your Parquet file
parquet_path = "/content/drive/MyDrive/complaint_embeddings.parquet"

# Open and process
parquet_file = pq.ParquetFile(parquet_path)
print(f"Row groups: {parquet_file.num_row_groups}")

# Initialize
index = None
dim = None
complaint_ids, products, chunk_texts, chunk_indices = [], [], [], []

for rg_idx in range(parquet_file.num_row_groups):
    table = parquet_file.read_row_group(rg_idx)
    df_batch = table.to_pandas()

    embeddings_batch = np.array(df_batch['embedding'].tolist(), dtype=np.float32)
    if dim is None:
        dim = embeddings_batch.shape[1]
        index = faiss.IndexFlatL2(dim)
    index.add(embeddings_batch)

    # Extract metadata from the 'metadata' column
    meta = df_batch['metadata']
    complaint_ids.extend([m['complaint_id'] for m in meta])
    products.extend([m['product_category'] for m in meta])
    chunk_indices.extend([m['chunk_index'] for m in meta])
    chunk_texts.extend(df_batch['document'].tolist())

    print(f"Processed RG {rg_idx+1}/{parquet_file.num_row_groups} – vectors: {index.ntotal}")

# Save local files
faiss.write_index(index, "full_faiss_index.bin")
metadata = {
    'complaint_ids': complaint_ids,
    'products': products,
    'chunk_texts': chunk_texts,
    'chunk_indices': chunk_indices
}
with open("full_metadata.pkl", 'wb') as f:
    pickle.dump(metadata, f)

print("✅ Local files saved.")

# Copy to Drive (backup)
!cp full_faiss_index.bin "/content/drive/MyDrive/full_faiss_index.bin"
!cp full_metadata.pkl "/content/drive/MyDrive/full_metadata.pkl"
print("✅ Files copied to Google Drive root.")

# Show sizes
import os
print("FAISS index size:", os.path.getsize("full_faiss_index.bin") / (1024**3), "GB")
print("Metadata size:", os.path.getsize("full_metadata.pkl") / (1024**3), "GB")

In [ ]:
import zipfile
with zipfile.ZipFile("full_vector_store.zip", "w") as zipf:
    zipf.write("full_faiss_index.bin")
    zipf.write("full_metadata.pkl")
from google.colab import files
files.download("full_vector_store.zip")

In [ ]:
import zipfile
with zipfile.ZipFile("full_vector_store.zip", "w") as zipf:
    zipf.write("full_faiss_index.bin")
    zipf.write("full_metadata.pkl")
from google.colab import files
files.download("full_vector_store.zip")

In [ ]:
!pip install -q faiss-cpu sentence-transformers pyarrow

In [ ]:
# ------------------------------------------------------------------
# 1. Load your FAISS index and metadata (if not already loaded)
# ------------------------------------------------------------------
import faiss
import pickle

# If you have the files locally
try:
    index = faiss.read_index("full_faiss_index.bin")
    with open("full_metadata.pkl", "rb") as f:
        metadata = pickle.load(f)
    print("✅ Index and metadata loaded.")
except FileNotFoundError:
    # If not, mount Drive and load from there
    from google.colab import drive
    drive.mount('/content/drive')
    !cp "/content/drive/MyDrive/full_faiss_index.bin" "/content/full_faiss_index.bin"
    !cp "/content/drive/MyDrive/full_metadata.pkl" "/content/full_metadata.pkl"
    index = faiss.read_index("full_faiss_index.bin")
    with open("full_metadata.pkl", "rb") as f:
        metadata = pickle.load(f)
    print("✅ Index and metadata loaded from Drive.")

# ------------------------------------------------------------------
# 2. Retriever function
# ------------------------------------------------------------------
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer('all-MiniLM-L6-v2', device='cuda')

def retrieve(query, top_k=5):
    q_emb = embed_model.encode(query, normalize_embeddings=True).astype('float32').reshape(1, -1)
    distances, indices = index.search(q_emb, top_k)
    results = []
    for i, idx in enumerate(indices[0]):
        results.append({
            'score': float(distances[0][i]),
            'complaint_id': metadata['complaint_ids'][idx],
            'product': metadata['products'][idx],
            'chunk_text': metadata['chunk_texts'][idx],
            'chunk_index': metadata['chunk_indices'][idx]
        })
    return results

# ------------------------------------------------------------------
# 3. Generator (LLM)
# ------------------------------------------------------------------
from huggingface_hub import InferenceClient

HF_TOKEN = "hf_your_token_here"   # Replace with your actual token
MODEL = "Qwen/Qwen2.5-7B-Instruct"
client = InferenceClient(model=MODEL, token=HF_TOKEN)

def ask_llm(prompt, max_new_tokens=300):
    response = client.chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=max_new_tokens,
    )
    return response.choices[0].message.content.strip()

# ------------------------------------------------------------------
# 4. Build Prompt
# ------------------------------------------------------------------
def build_prompt(question, retrieved_chunks):
    context = ""
    for i, chunk in enumerate(retrieved_chunks, 1):
        context += f"\n[Excerpt {i} | Product: {chunk['product']}]\n{chunk['chunk_text']}\n"

    prompt = f"""You are a financial analyst assistant for CreditTrust.
Use ONLY the following complaint excerpts to answer the question.
If the excerpts do not contain enough information, say exactly: "I don't have enough information to answer that."
Do not add any information not present in the excerpts.

Retrieved excerpts:
{context}

Question: {question}

Answer:"""
    return prompt

# ------------------------------------------------------------------
# 5. Full RAG Pipeline
# ------------------------------------------------------------------
def rag_pipeline(question, top_k=5):
    retrieved = retrieve(question, top_k)
    prompt = build_prompt(question, retrieved)
    answer = ask_llm(prompt)
    return {
        'question': question,
        'answer': answer,
        'sources': retrieved
    }

print("✅ RAG pipeline ready!")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy to Drive (if not already)
!cp full_faiss_index.bin "/content/drive/MyDrive/full_faiss_index.bin"
!cp full_metadata.pkl "/content/drive/MyDrive/full_metadata.pkl"

print("✅ Files copied to Drive.")

In [ ]:
from huggingface_hub import InferenceClient

HF_TOKEN = os.getenv(HF_TOKEN, "hf_tokenHere")
client = InferenceClient(model="Qwen/Qwen2.5-7B-Instruct", token=HF_TOKEN)

response = client.chat_completion(
    messages=[{"role": "user", "content": "Say 'Hello' in one word."}],
    max_tokens=10,
)
print(response.choices[0].message.content)

In [ ]:
test = rag_pipeline("Why are customers unhappy with credit cards?")
print("Answer:", test['answer'])
print("\nSources:")
for s in test['sources'][:2]:
    print(f"  - {s['product']} (score: {s['score']:.3f})")

In [ ]:
test_questions = [
    "Why are customers unhappy with credit cards?",
    "What are the most common complaints about personal loans?",
    "Are there any fraud‑related complaints?",
    "What issues do customers face with money transfers?",
    "How do customers describe billing disputes?",
    "What is the overall sentiment about savings accounts?",
    "Which products have the most complaints about fees?",
    "What actions have companies taken in response to complaints?",
    "Are there any complaints about customer service?",
    "What is the weather forecast for Nairobi tomorrow?"   # Out-of-scope test
]

for i, q in enumerate(test_questions, 1):
    result = rag_pipeline(q)
    print(f"\n=== Q{i}: {q} ===")
    print(f"Answer: {result['answer']}")
    print("Sources:")
    for s in result['sources'][:2]:
        print(f"  - {s['product']} (score: {s['score']:.3f})")
    print("-" * 60)